# 2. Splitting data without leakage

## Small idea: the split should imitate future use

Random row splitting is appropriate only when rows are genuinely independent and future
observations come from the same distribution. Learner corpora, repeated measurements,
temporal corpora, and duplicated texts often require group- or time-aware splits.

**Learning goals**

- create train, validation, and test partitions;
- use stratification for independent rows;
- keep repeated learners and duplicate clusters together;
- use chronological splits when predicting the future;
- identify common forms of leakage.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split

rng = np.random.default_rng(42)
learners = [f"L{i:02d}" for i in range(30)]
learner_level = dict(zip(learners, rng.choice(["A2", "B1", "B2"], size=len(learners), p=[0.25, 0.5, 0.25])))

rows = []
for learner in learners:
    for response_number in range(2):
        rows.append({
            "response_id": f"{learner}_R{response_number + 1}",
            "learner_id": learner,
            "proficiency": learner_level[learner],
            "tokens": int(rng.integers(60, 240)),
            "year": int(rng.integers(2022, 2027)),
        })

data = pd.DataFrame(rows)
data.head()

## 1. Why a row split can fail

A normal random split can place one learner's first response in training and the same
learner's second response in testing. The test set is then not independent.

In [ ]:
row_train, row_test = train_test_split(data, test_size=0.25, random_state=42)
learner_overlap = set(row_train["learner_id"]) & set(row_test["learner_id"])

print("Learners appearing in both partitions:", len(learner_overlap))
print("Example overlap:", sorted(learner_overlap)[:5])

## 2. A three-way group split

`GroupShuffleSplit` treats a learner as an indivisible unit. The first split creates
training and temporary partitions; the second divides the temporary groups into
validation and test partitions.

In [ ]:
outer_split = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=42)
train_idx, temp_idx = next(
    outer_split.split(data, y=data["proficiency"], groups=data["learner_id"])
)

train = data.iloc[train_idx].reset_index(drop=True)
temp = data.iloc[temp_idx].reset_index(drop=True)

inner_split = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=43)
validation_idx, test_idx = next(
    inner_split.split(temp, y=temp["proficiency"], groups=temp["learner_id"])
)

validation = temp.iloc[validation_idx].reset_index(drop=True)
test = temp.iloc[test_idx].reset_index(drop=True)

print("Rows:", {"train": len(train), "validation": len(validation), "test": len(test)})

In [ ]:
train_groups = set(train["learner_id"])
validation_groups = set(validation["learner_id"])
test_groups = set(test["learner_id"])

assert train_groups.isdisjoint(validation_groups)
assert train_groups.isdisjoint(test_groups)
assert validation_groups.isdisjoint(test_groups)
print("Learner groups are disjoint.")

In [ ]:
split_balance = pd.concat({
    "train": train["proficiency"].value_counts(normalize=True),
    "validation": validation["proficiency"].value_counts(normalize=True),
    "test": test["proficiency"].value_counts(normalize=True),
}, axis=1).fillna(0).round(2)
split_balance

Group splitting does not automatically guarantee exact class proportions. Report the
distribution and, when both grouping and class balance are essential, use a documented
group-aware stratification strategy during Stage 6 cross-validation.

## 3. Stratification for independent rows

In [ ]:
independent = pd.DataFrame({
    "item_id": [f"I{i:03d}" for i in range(100)],
    "label": [0] * 80 + [1] * 20,
})

independent_train, independent_test = train_test_split(
    independent,
    test_size=0.20,
    random_state=42,
    stratify=independent["label"],
)

print("Full positive rate:", independent["label"].mean())
print("Train positive rate:", independent_train["label"].mean())
print("Test positive rate:", independent_test["label"].mean())

## 4. Chronological splitting

If the deployment question is “Will a model trained on the past work on future data?”,
sort by time and place the newest period in validation/test. Never shuffle future rows
into training.

In [ ]:
monthly = pd.DataFrame({
    "month": pd.date_range("2024-01-01", periods=24, freq="MS"),
    "document_count": rng.integers(80, 150, size=24),
})

chronological_train = monthly[monthly["month"] < "2025-07-01"]
chronological_test = monthly[monthly["month"] >= "2025-07-01"]

print(chronological_train["month"].min(), "to", chronological_train["month"].max())
print(chronological_test["month"].min(), "to", chronological_test["month"].max())

## 5. Duplicate and template clusters

Exact duplicates, near-duplicates, reposts, translations, and shared writing prompts can
leak across partitions. Assign a cluster or template ID and use it as a split group. If
both learner and duplicate constraints exist, create groups that keep every connected
learner/duplicate component together.

In [ ]:
clustered = pd.DataFrame({
    "text": ["متن الف", "متن الف!", "متن ب", "متن ج", "متن ج"],
    "duplicate_cluster": ["C1", "C1", "C2", "C3", "C3"],
})

cluster_split = GroupShuffleSplit(n_splits=1, train_size=0.60, random_state=7)
cluster_train_idx, cluster_test_idx = next(
    cluster_split.split(clustered, groups=clustered["duplicate_cluster"])
)

assert set(clustered.iloc[cluster_train_idx]["duplicate_cluster"]).isdisjoint(
    set(clustered.iloc[cluster_test_idx]["duplicate_cluster"])
)
print("Duplicate clusters are disjoint.")

## Leakage checklist

- Fit imputers, encoders, scalers, vectorizers, and selectors on training data only.
- Never choose transformations or hyperparameters by repeatedly checking the test set.
- Remove target-derived and post-outcome columns.
- Keep repeated participants, sites, conversations, documents, and duplicate clusters together.
- Use time order when the intended use is future prediction.
- Lock the test set until the final evaluation.

## Tiny checkpoint

Choose the split unit for: repeated essays from one learner, sentences sampled from the
same document, social-media reposts, and monthly data used to predict next year's labels.